<a href="https://colab.research.google.com/github/ggug0125-ui/AI_26/blob/main/AI_%EA%B8%B0%EB%B0%98%EC%98%88%EC%B8%A1%EB%B0%8F%EB%B6%84%EC%84%9D(%EC%9E%84%ED%9A%A8%EC%A0%95).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 전처리
import tensorflow as tf  # 텐서플로우 불러오기
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
import matplotlib.pyplot as plt # 그래프와 이미지 출력을 위한 라이브러리 불러오기
import numpy as np  # 배열 계산을 위한 넘파이 불러오기

from tensorflow import keras # 텐서플로우 안에 케라스 불러오기
from sklearn.model_selection import train_test_split # 데이터를 훈련용 / 검증용으로 나누는 함수 불러오기

(train_input, train_target), (test_input, test_target) = \
    keras.datasets.fashion_mnist.load_data()
train_scaled = train_input.reshape(-1, 28, 28, 1) / 255.0
test_scaled = test_input.reshape(-1, 28, 28, 1) / 255.0

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
print("훈련 데이터 :", train_scaled.shape, train_target.shape)
print("검증 데이터 :", val_scaled.shape, val_target.shape)
print("테스트 데이터 :", test_scaled.shape, test_target.shape)

훈련 데이터 : (48000, 28, 28, 1) (48000,)
검증 데이터 : (12000, 28, 28, 1) (12000,)
테스트 데이터 : (10000, 28, 28, 1) (10000,)


In [3]:
model = keras.Sequential()   # 모델 생성
model.add(keras.layers.Conv2D(   # 합성곱 층 추가 32필터사용 3*3
    32, kernel_size=3, activation='relu', padding='same', input_shape=(28, 28, 1)))
model.add(keras.layers.MaxPooling2D(2)) # 최대 풀링층 추가
model.add(keras.layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'))
# 합성곱층 두번째 추가 64개필터
model.add(keras.layers.MaxPooling2D(2)) # 최대 풀링층 추가
model.add(keras.layers.Flatten())  # 2차원 특징맵을 1차원 일렬로 펼침
model.add(keras.layers.Dense(128, activation='relu'))
# 은닉층 128개 뉴런사용, 활성화함수는 렐루
model.add(keras.layers.Dropout(0.3))
 # 드롭아웃사용, 과대적합막아보기 뉴런의 30% 랜덤하게 꺼냄
model.add(keras.layers.Dense(10, activation='softmax'))
# 출력층 추가 확률계산
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005),  # 모델학습
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
checkpoint_cb = keras.callbacks.ModelCheckpoint('best-cnn-model.keras',   # 가장성능좋은모델만 저장
                                                save_best_only=True)
early_stopping_cb = keras.callbacks.EarlyStopping(patience=3,
                                                  restore_best_weights=True)  # 검증3번정도 상향시 조기종료
history = model.fit(train_scaled, train_target, epochs=30,     # 학습 시작 에포크 30번 학습해보자
                    validation_data=(val_scaled, val_target),
                    callbacks=[checkpoint_cb, early_stopping_cb])


Epoch 1/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 79s 51ms/step - accuracy: 0.7388 - loss: 0.7288 - val_accuracy: 0.8752 - val_loss: 0.3381
Epoch 2/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 71s 47ms/step - accuracy: 0.8720 - loss: 0.3566 - val_accuracy: 0.8894 - val_loss: 0.2971
Epoch 3/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 70s 47ms/step - accuracy: 0.8895 - loss: 0.3055 - val_accuracy: 0.9012 - val_loss: 0.2668
Epoch 4/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 87s 50ms/step - accuracy: 0.9020 - loss: 0.2687 - val_accuracy: 0.9091 - val_loss: 0.2464
Epoch 5/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 73s 49ms/step - accuracy: 0.9086 - loss: 0.2464 - val_accuracy: 0.9144 - val_loss: 0.2372
Epoch 6/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 79s 52ms/step - accuracy: 0.9177 - loss: 0.2237 - val_accuracy: 0.9126 - val_loss: 0.2427
Epoch 7/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 82s 52ms/step - accuracy: 0.9222 - loss: 0.2068 - val_accuracy: 0.9210 - val_loss: 0.2240
Epoch 8/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 74s 49ms/step - accuracy: 0.9280 -

In [ ]:
# 그래프 보기
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.xlabel('epoch')
plt.ylabel('loss / accuracy')
plt.legend(['train_loss', 'val_loss', 'train_accuracy', 'val_accuracy'])
plt.show()

In [ ]:
model.evaluate(val_scaled, val_target)  # 검증 데이터로 모델 성능 평가

In [ ]:
model.evaluate(test_scaled, test_target)  # 테스트 데이터로 모델 최종 성능 평가

In [ ]:
plt.imshow(val_scaled[0].reshape(28, 28), cmap='gray_r')   # 검증 0번째 이미
plt.show()

In [ ]:
preds = model.predict(val_scaled[0:1])
print(preds)  # 예측활률 출

In [ ]:
# 그래프보기
plt.bar(range(0, 10), preds[0])
plt.xlabel('class')
plt.ylabel('prob.')
plt.show()

In [ ]:
classes = ['티셔츠', '바지', '스웨터', '드레스', '코트',
           '샌달', '셔츠', '스니커즈', '가방', '앵클 부츠']

In [ ]:
print(classes[np.argmax(preds)])

In [ ]:
preds = model.predict(test_scaled[15:16])  # 테스트 15번째 이미지보

In [ ]:
print(preds)   # 예측확률 출력

In [ ]:
plt.bar(range(0, 10), preds[0])

In [ ]:
print(classes[np.argmax(preds)])